In [2]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# 加载模型和分词器
model_path = "D:/PycharmProjects/TDFilter/textclassific/Biological_roberta_10_5/checkpoint-280"  # 替换为你的模型路径
model = AutoModelForSequenceClassification.from_pretrained(model_path)
tokenizer = AutoTokenizer.from_pretrained(model_path)


In [3]:

from datasets import load_dataset, DatasetDict

# 加载 JSON 数据集
dataset = load_dataset('json', data_files='D:/PycharmProjects/TDFilter/DataSet/Science/test_dataset.json')


Generating train split: 0 examples [00:00, ? examples/s]

In [4]:
dataset

DatasetDict({
    train: Dataset({
        features: ['flag', 'generateKnowledge', 'basicKnowledge'],
        num_rows: 16400
    })
})

In [6]:
# 3. 提取 'generateKnowledge' 字段的数据（假设数据位于 'train' 分区中）
generate_knowledge_texts = dataset['train']['generateKnowledge']
inputs = tokenizer(generate_knowledge_texts, padding=True, truncation=True, return_tensors="pt")


In [1]:
import json
import torch
import numpy as np
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# 1. 加载模型和分词器
model_path = "D:/PycharmProjects/TDFilter/textclassific/Science_roberta91/checkpoint-64"  # 替换为你的模型路径
model = AutoModelForSequenceClassification.from_pretrained(model_path)
tokenizer = AutoTokenizer.from_pretrained(model_path)

# 2. 从 JSON 文件加载测试数据
with open('D:/PycharmProjects/TDFilter/DataSet/Science/test_dataset.json', 'r') as f:
    test_data=[]
    for line in f:
        test_data.append(json.loads(line.strip(",\n")))


# 3. 遍历所有数据并提取 generateKnowledge 和 flag 字段
generate_knowledge_texts = [item["generateKnowledge"] for item in test_data]
true_flags = [item["flag"] for item in test_data]  # 提取真实的 flag 值

# 4. 对所有提取的文本进行分词处理
inputs = tokenizer(generate_knowledge_texts, padding=True, truncation=True, return_tensors="pt")

# 5. 使用模型进行推理
with torch.no_grad():
    outputs = model(**inputs)

# 6. 获取 logits 并将其转换为预测类别
logits = outputs.logits
predicted_classes = torch.argmax(logits, dim=-1).tolist()

# 7. 输出所有预测结果
# for i, (text, prediction, true_flag) in enumerate(zip(generate_knowledge_texts, predicted_classes, true_flags)):
#     print(f"Text {i+1}: Predicted flag = {prediction}, True flag = {true_flag}")

# 8. 计算准确率
def compute_accuracy(predictions, labels):
    return np.mean(np.array(predictions) == np.array(labels))

accuracy = compute_accuracy(predicted_classes, true_flags)
print(f"Accuracy: {accuracy * 100:.2f}%")



KeyboardInterrupt



In [1]:
import json
import torch
import numpy as np
import time  # 导入 time 模块
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# 1. 加载模型和分词器
model_path = "D:/PycharmProjects/TDFilter/textclassific/Biological_roberta_10_5/checkpoint-280" # 替换为你的模型路径
model = AutoModelForSequenceClassification.from_pretrained(model_path)
tokenizer = AutoTokenizer.from_pretrained(model_path)

# 2. 从 JSON 文件加载测试数据
with open('D:/PycharmProjects/TDFilter/DataSet/Biology/test_dataset_10_5.json', 'r',encoding='utf-8') as f:
    test_data=[]
    for line in f:
        test_data.append(json.loads(line.strip(",\n")))


# 3. 提取 generateKnowledge 和 flag 字段
generate_knowledge_texts = [item["generateKnowledge"] for item in test_data]
true_flags = [item["flag"] for item in test_data]  # 提取真实的 flag 值

# 4. 定义批处理大小和结果列表
batch_size = 32  # 可以根据你的硬件情况调整批次大小
predicted_classes = []

# 定义输出文件路径
output_file = 'D:/PycharmProjects/TDFilter/DataSet/Biology/test_dataset_10_5_basic_predict.json'  # 使用 .jsonl 扩展名表示 JSON Lines 格式

# 5. 分批处理数据
for i in range(0, len(generate_knowledge_texts), batch_size):
    batch_texts = generate_knowledge_texts[i:i+batch_size]
    batch_labels = true_flags[i:i+batch_size]

    # 对当前批次的文本进行分词处理
    inputs = tokenizer(batch_texts, padding=True, truncation=True, return_tensors="pt")

    # 使用模型进行推理
    with torch.no_grad():
        outputs = model(**inputs)

    # 获取 logits 并转换为预测类别
    logits = outputs.logits
    batch_predictions = torch.argmax(logits, dim=-1).tolist()
    predicted_classes.extend(batch_predictions)

    # 将预测结果和真实标签以 JSON 格式写入文件
    with open(output_file, 'a', encoding='utf-8') as f_output:
        for text, prediction, true_flag in zip(batch_texts, batch_predictions, batch_labels):
            json_line = json.dumps({
                'text': text,
                'predicted_flag': prediction,
                'true_flag': true_flag
            }, ensure_ascii=False)
            f_output.write(json_line + '\n')  # 每个 JSON 对象占一行

    # 在每个批次之间加入延迟
    time.sleep(0.05)  # 延迟 1 秒，可以根据需要调整

# 6. 计算准确率
def compute_accuracy(predictions, labels):
    return np.mean(np.array(predictions) == np.array(labels))

accuracy = compute_accuracy(predicted_classes, true_flags)
print(f"Accuracy: {accuracy * 100:.2f}%")

# 将准确率也写入文件
with open(output_file, 'a', encoding='utf-8') as f_output:
    json_line = json.dumps({'Accuracy': f"{accuracy * 100:.2f}%"}, ensure_ascii=False)
    f_output.write(json_line + '\n')


Accuracy: 87.99%


In [2]:
# 用于计算混淆矩阵
import json
import pandas as pd
from sklearn.metrics import confusion_matrix

# 读取 JSONL 文件
def load_jsonl(file_path):
    data = []
    with open(file_path, 'r', encoding='utf-8') as file:
        for line in file:
            data.append(json.loads(line))
    return data

# 计算混淆矩阵
def compute_confusion_matrix(data):
    # 将数据转换为 DataFrame
    df = pd.DataFrame(data)
    # 删除包含 NaN 的行
    df = df.dropna(subset=['true_flag', 'predicted_flag'])
    # 计算混淆矩阵
    cm = confusion_matrix(df['true_flag'], df['predicted_flag'])

    return cm

# 指定 JSONL 文件路径
file_path = output_file

# 加载数据
data = load_jsonl(file_path)

# 计算混淆矩阵
cm = compute_confusion_matrix(data)

# 输出混淆矩阵
print("Confusion Matrix:")
print(cm)
# (准确率为）


Confusion Matrix:
[[13967  2512]
 [ 2074 19642]]


In [3]:
# roberta-base  矛盾推理能力
import torch
from transformers import RobertaTokenizer, RobertaForSequenceClassification

# 加载 RoBERTa 模型和分词器
model_name = 'roberta-base'
tokenizer = RobertaTokenizer.from_pretrained(model_name)
model = RobertaForSequenceClassification.from_pretrained(model_name, num_labels=3)  # 3个标签：蕴含、矛盾、中立

# 输入句子
premise = "The cat is sleeping on the mat."
hypothesis = "The cat is not on the mat."

# 编码输入
inputs = tokenizer(premise, hypothesis, return_tensors='pt', padding=True, truncation=True)# 模型推理
with torch.no_grad():
    outputs = model(**inputs)

# 获取预测结果
logits = outputs.logits
predicted_label = torch.argmax(logits, dim=1).item()

# 标签映射
labels = {0: 'Entailment', 1: 'Contradiction', 2: 'Neutral'}
print(f"Predicted relationship: {labels[predicted_label]}")


Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Predicted relationship: Entailment


In [23]:
import json
from sklearn.metrics import accuracy_score

file_path = 'D:/PycharmProjects/TDFilter/textclassific/contradiction_biological_roberta_trueEnviroment/checkpoint-1500'

class ContradictionInferencer:
    def __init__(self):
        # 初始化模型，例如加载预训练的 RoBERTa 模型
        from transformers import RobertaTokenizer, RobertaForSequenceClassification
        import torch

        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        self.tokenizer = RobertaTokenizer.from_pretrained(file_path)
        self.model = RobertaForSequenceClassification.from_pretrained(file_path)
        self.model.to(self.device)
        self.model.eval()

    def infer(self, premise_list, hypothesis_list):
        import torch

        # 批量推理
        inputs = self.tokenizer(premise_list, hypothesis_list, return_tensors='pt', truncation=True, padding=True, max_length=512)
        inputs = {k: v.to(self.device) for k, v in inputs.items()}
        with torch.no_grad():
            outputs = self.model(**inputs)
        logits = outputs.logits
        predictions = torch.argmax(logits, dim=-1).tolist()
        return predictions

if __name__ == "__main__":
    # 创建推理器实例
    inferencer = ContradictionInferencer()

    # 数据集文件路径
    data_file = 'D:/PycharmProjects/TDFilter/experiment4/roberta_iterate/results.jsonl'  # 数据集文件路径
    output_file = 'prediction_results_trueEnviroment.txt'  # 输出文件路径

    batch_size = 32  # 您可以根据内存大小调整批次大小

    total_samples = 0
    correct_predictions = 0

    basic_knowledge_batch = []
    generate_knowledge_batch = []
    true_labels_batch = []

    with open(data_file, 'r', encoding='utf-8') as f_data, open(output_file, 'w', encoding='utf-8') as f_output:
        for line in f_data:
            data = json.loads(line.strip())
            basic_knowledge_batch.append(data['knowledge'])
            generate_knowledge_batch.append(data['matchingKnowledge'])
            true_labels_batch.append(data['flag'])

            if len(basic_knowledge_batch) == batch_size:
                # 进行推理
                predicted_labels = inferencer.infer(basic_knowledge_batch, generate_knowledge_batch)

                # 计算准确率
                batch_total = len(true_labels_batch)
                batch_correct = sum([1 for pred, true in zip(predicted_labels, true_labels_batch) if pred == true])
                total_samples += batch_total
                correct_predictions += batch_correct

                # 写入结果
                for idx, (pred, true) in enumerate(zip(predicted_labels, true_labels_batch)):
                    sample_id = total_samples - batch_total + idx + 1
                    pred_result = '不矛盾' if pred == 1 else '矛盾'
                    true_result = '不矛盾' if true == 1 else '矛盾'
                    correctness = '✅' if pred == true else '❌'
                    result_line = f"样本 {sample_id}: 预测结果={pred_result}，真实标签={true_result} {correctness}\n"
                    f_output.write(result_line)

                # 清空批次列表
                basic_knowledge_batch = []
                generate_knowledge_batch = []
                true_labels_batch = []

        # 处理最后一个不满批次的样本
        if basic_knowledge_batch:
            predicted_labels = inferencer.infer(basic_knowledge_batch, generate_knowledge_batch)
            batch_total = len(true_labels_batch)
            batch_correct = sum([1 for pred, true in zip(predicted_labels, true_labels_batch) if pred == true])
            total_samples += batch_total
            correct_predictions += batch_correct

            for idx, (pred, true) in enumerate(zip(predicted_labels, true_labels_batch)):
                sample_id = total_samples - batch_total + idx + 1
                pred_result = '不矛盾' if pred == 1 else '矛盾'
                true_result = '不矛盾' if true == 1 else '矛盾'
                correctness = '✅' if pred == true else '❌'
                result_line = f"样本 {sample_id}: 预测结果={pred_result}，真实标签={true_result} {correctness}\n"
                f_output.write(result_line)

    # 最终计算总体准确率
    accuracy = correct_predictions / total_samples
    print(f"\n模型准确率: {accuracy * 100:.2f}%\n")
    print(f"预测结果已保存到文件：{output_file}")



模型准确率: 95.37%

预测结果已保存到文件：prediction_results_trueEnviroment.txt


In [1]:
import json
from sklearn.metrics import accuracy_score
file_path = 'D:/PycharmProjects/TDFilter/textclassific/biologicExtend_roberta/checkpoint-419'
class ContradictionInferencer:
    def __init__(self):
        # 初始化模型，例如加载预训练的 RoBERTa 模型
        from transformers import RobertaTokenizer, RobertaForSequenceClassification
        import torch

        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        self.tokenizer = RobertaTokenizer.from_pretrained(file_path)
        self.model = RobertaForSequenceClassification.from_pretrained(file_path)
        self.model.to(self.device)
        self.model.eval()

    def infer(self, premise_list, hypothesis_list):
        import torch
 # 批量推理
        inputs = self.tokenizer(premise_list, hypothesis_list, return_tensors='pt', truncation=True, padding=True, max_length=512)
        inputs = {k: v.to(self.device) for k, v in inputs.items()}
        with torch.no_grad():
            outputs = self.model(**inputs)
        logits = outputs.logits
        predictions = torch.argmax(logits, dim=-1).tolist()
        return predictions

if __name__ == "__main__":
    # 创建推理器实例
    inferencer = ContradictionInferencer()

    # 加载数据集
    data_file = 'D:/PycharmProjects/TDFilter/DataSet/biology/biologicExtend.json'  # 数据集文件路径
    output_file = 'D:/PycharmProjects/TDFilter/DataSet/biology/prediction_results_trueEnviroment.txt'  # 输出文件路径
    batch_size = 32  # 您可以根据内存大小调整批次大小

    total_samples = 0
    correct_predictions = 0

    basic_knowledge_batch = []
    generate_knowledge_batch = []
    true_labels_batch = []

    with open(data_file, 'r', encoding='utf-8') as f_data:
        data = json.load(f_data)

    basic_knowledge_list = [item['basicKnoledge'] for item in data]
    generate_knowledge_list = [item['generateKnowledge'] for item in data]
    true_labels = [item['flag'] for item in data]

    # 批次大小
    batch_size = 32  # 您可以根据内存大小调整批次大小

    total_samples = len(data)
    correct_predictions = 0

    with open(output_file, 'w', encoding='utf-8') as f_output:
        for start_idx in range(0, total_samples, batch_size):
            end_idx = min(start_idx + batch_size, total_samples)
            basic_knowledge_batch = basic_knowledge_list[start_idx:end_idx]
            generate_knowledge_batch = generate_knowledge_list[start_idx:end_idx]
            true_labels_batch = true_labels[start_idx:end_idx]

            # 进行推理
            predicted_labels = inferencer.infer(basic_knowledge_batch, generate_knowledge_batch)

            # 计算准确率
            batch_correct = sum([1 for pred, true in zip(predicted_labels, true_labels_batch) if pred == true])
            correct_predictions += batch_correct

            # 写入结果
            for idx, (pred, true) in enumerate(zip(predicted_labels, true_labels_batch)):
                sample_id = start_idx + idx + 1
                pred_result = '不矛盾' if pred == 1 else '矛盾'
                true_result = '不矛盾' if true == 1 else '矛盾'
                correctness = '✅' if pred == true else '❌'
                result_line = f"样本 {sample_id}: 预测结果={pred_result}，真实标签={true_result} {correctness}\n"
                f_output.write(result_line)

    # 最终计算总体准确率
    accuracy = correct_predictions / total_samples
    print(f"\n模型准确率: {accuracy * 100:.2f}%\n")
    print(f"预测结果已保存到文件：{output_file}")

KeyError: 'basicKnoledge'